# Experiment Tracking with MLflow — Module 1 Foundations

This notebook demonstrates experiment tracking for ML model development:
- MLflow setup and experiment management
- Parameter, metric, and artifact logging
- Model comparison and selection
- Hyperparameter tuning tracking
- Model registry basics

**Learning Outcomes:**
1. Track experiments reproducibly
2. Compare models across runs
3. Log and version models
4. Transition models through stages

In [ ]:
# Imports and MLflow Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# MLflow
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Set up local tracking
mlflow.set_tracking_uri("file:./mlruns")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Create or get experiment
experiment_name = "adult-income-classification"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Created experiment: {experiment_name} (ID: {experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment: {experiment_name} (ID: {experiment_id})")

mlflow.set_experiment(experiment_name)

In [ ]:
# Load and Prepare Data
adult = fetch_openml(data_id=1590, as_frame=True, parser='pandas')
df = adult.frame.copy()

target_col = 'class'
feature_cols = [c for c in df.columns if c != target_col]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

# Preprocessing pipeline (shared across models)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features)
    ]
)

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Define Models to Compare
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVC": SVC(probability=True, random_state=42)
}

def train_and_log_model(model_name, model, X_train, y_train, X_test, y_test, params=None):
    """Train a model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name) as run:
        # Build full pipeline
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        
        # Log parameters
        mlflow.log_param("model_type", model_name)
        if params:
            for k, v in params.items():
                mlflow.log_param(k, v)
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("test_size", len(X_test))
        
        # Train
        pipeline.fit(X_train, y_train)
        
        # Predict
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
        
        # Calculate metrics
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average='weighted'),
            "recall": recall_score(y_test, y_pred, average='weighted'),
            "f1": f1_score(y_test, y_pred, average='weighted'),
        }
        if y_proba is not None:
            metrics["roc_auc"] = roc_auc_score(y_test, y_proba)
        
        # Log metrics
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
        
        # Log model
        mlflow.sklearn.log_model(pipeline, "model", registered_model_name=f"AdultIncome_{model_name}")
        
        # Log cross-validation scores
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
        mlflow.log_metric("cv_accuracy_mean", cv_scores.mean())
        mlflow.log_metric("cv_accuracy_std", cv_scores.std())
        
        print(f"{model_name}: Accuracy={metrics['accuracy']:.4f}, F1={metrics['f1']:.4f}, CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}")
        
        return run.info.run_id, pipeline, metrics

print("Starting model comparison experiment...")

In [ ]:
# Run Model Comparison
results = {}
for name, model in models.items():
    run_id, pipeline, metrics = train_and_log_model(
        name, model, X_train, y_train, X_test, y_test
    )
    results[name] = {"run_id": run_id, "pipeline": pipeline, "metrics": metrics}

print("\n=== Model Comparison Complete ===")
for name, res in results.items():
    m = res['metrics']
    print(f"{name:20s} | Acc: {m['accuracy']:.4f} | F1: {m['f1']:.4f} | ROC-AUC: {m.get('roc_auc', 'N/A')}")

In [ ]:
# Hyperparameter Tuning with MLflow Tracking
from sklearn.model_selection import GridSearchCV

# Define parameter grid for Random Forest
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5, 10]
}

print("Running GridSearchCV with MLflow tracking...")

with mlflow.start_run(run_name="RandomForest_GridSearch") as run:
    # Base pipeline
    rf_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
    ])
    
    # Grid search
    grid_search = GridSearchCV(
        rf_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Log best parameters
    for param, value in grid_search.best_params_.items():
        mlflow.log_param(param, value)
    
    # Log best score
    mlflow.log_metric("best_cv_accuracy", grid_search.best_score_)
    
    # Evaluate on test set
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    test_metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, average='weighted'),
        "test_recall": recall_score(y_test, y_pred, average='weighted'),
        "test_f1": f1_score(y_test, y_pred, average='weighted'),
    }
    
    for k, v in test_metrics.items():
        mlflow.log_metric(k, v)
    
    # Log best model
    mlflow.sklearn.log_model(best_model, "best_model", registered_model_name="AdultIncome_RF_Tuned")
    
    print(f"\nBest params: {grid_search.best_params_}")
    print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
    print(f"Test metrics: {test_metrics}")

In [ ]:
# Query and Compare Runs
client = MlflowClient()

# Get all runs for this experiment
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=20
)

print(f"Found {len(runs)} runs\n")
print(f"{'Run Name':<30} {'Model':<20} {'Accuracy':>10} {'F1':>10} {'CV Acc':>10}")
print("-" * 85)

for run in runs:
    name = run.data.tags.get('mlflow.runName', 'unnamed')
    model = run.data.params.get('model_type', 'unknown')
    acc = run.data.metrics.get('accuracy', 0)
    f1 = run.data.metrics.get('f1', 0)
    cv_acc = run.data.metrics.get('cv_accuracy_mean', 0)
    print(f"{name:<30} {model:<20} {acc:>10.4f} {f1:>10.4f} {cv_acc:>10.4f}")

# Get best model from registry
print("\n=== Registered Models ===")
for rm in client.search_registered_models():
    print(f"\n{rm.name}:")
    for mv in client.search_model_versions(f"name='{rm.name}'"):
        stage = mv.current_stage
        print(f"  v{mv.version} - Stage: {stage} - Run: {mv.run_id[:8]}")

## Launch MLflow UI

To view the experiment tracking UI, run in terminal:

```bash
mlflow ui --backend-store-uri file:./mlruns --port 5000
```

Then open http://localhost:5000 in your browser.

## Summary

**What we covered:**
✓ MLflow experiment setup and tracking
✓ Logging parameters, metrics, and models
✓ Comparing multiple algorithms in one experiment
✓ Hyperparameter tuning with GridSearchCV + MLflow
✓ Model registry basics (registering, versioning, stages)
✓ Querying runs programmatically

**Key MLflow Concepts:**
- **Experiment**: Container for related runs
- **Run**: Single execution with params, metrics, artifacts
- **Artifact**: Files logged during a run (models, plots, data)
- **Model Registry**: Centralized model store with versioning and stages (Staging, Production, Archived)

**Next Steps:**
- Add custom metrics (fairness, latency, business KPIs)
- Log data lineage and dataset versions
- Set up automated model promotion workflows
- Integrate with CI/CD for deployment gates
- Try MLflow Projects for reproducible runs